In [1]:
import numba as nb
import numpy as np
import seaborn as sns
import sympy as sp
from pprint import pprint

from auxin_signalling_models import \
       ReducedAuxinSignallingPathway as asp_class


In [3]:
# Generate a simple model with 1 ARF and 2 Aux/Iaas
asp_model = asp_class(nARFs=2, nIAAs=1)
empty_set_node = asp_model.empty_set_node


#
tas3_gene_var = "TAS3"

asp_model.add_state(tas3_gene_var, category="RNA", initial_condition=0.1)
asp_model.add_state("miR390", category="RNA", initial_condition=0.1)

tasi_prod_rate = asp_model.add_model_parameter("k__TAS3", 0.1, "rate of tasi-arf mediated production")
tasi_decay_rate = asp_model.add_model_parameter("d__TAS3", 0.1, "rate of tasi-arf mediated decay")
decay_rate_mirna = asp_model.add_model_parameter("d__mR390", 0.1, "miR390 microRNA decay")


# Redo this
prod_rate1 = asp_model.add_model_parameter("k__G__ARF_1__ARF_1__R__miR390", 1.0, "TODO")
prod_rate2 = asp_model.add_model_parameter("k__R__miR390", 0.1, "Rate of production of miR390 from unbound promoter")
prod_rate3 = asp_model.add_model_parameter("k__miR390_tasi_arf", 0.1, "Rate of production of ta-si ARF from miRNA")


# Add production reactions
asp_model.add_reaction([], ["miR390"], [prod_rate1.name], [], enzymes=["G__ARF_1__ARF_1"])
asp_model.add_reaction([], ["miR390"], [prod_rate2.name], [], enzymes=["G"])
asp_model.add_reaction([], [tas3_gene_var], [tasi_prod_rate.name], [tasi_decay_rate.name],
                       enzymes=["miR390"])
asp_model.add_reaction(["miR390"], [empty_set_node.name], [decay_rate_mirna.name], [])


tasi_R_ARF_prod_rate = asp_model.add_model_parameter("k__tasi_R_ARF", 0.1,
                                                     "rate at which miR390 turns TAS3 transcripts into tasi-RARF")

tasi_decay_rate = asp_model.add_model_parameter("d__ARF_2__tasi", 1.0)
asp_model.add_reaction(["ARF_2"], [empty_set_node.name], [tasi_decay_rate.name], [], enzymes=[tas3_gene_var])

tasi_arf_var = "tasi_R_ARF"
asp_model.add_state(tasi_arf_var)

tasi_R_ARF_decay = asp_model.add_model_parameter("d__tasi_R_ARF_decay", 1.0, "Decay of tasi RARF")

asp_model.add_reaction([tas3_gene_var], [tasi_arf_var], [tasi_R_ARF_prod_rate.name], [],
                      enzymes = ["miR390"])
asp_model.add_reaction([tasi_arf_var], [empty_set_node.name], [tasi_R_ARF_decay.name], [])

# Output default initial conditions and parameters
print('default initial conditions:')
pprint(asp_model.get_default_initial_conditions())

print("Model state variables")
print(asp_model.get_state_variables())

print("Model parameters")
print(asp_model.get_parameters())

print('\n\ndefault parameter values:')
pprint(asp_model.get_default_parameters())

default_parameters = asp_model.get_default_parameters()
default_initial_conditions = asp_model.get_default_initial_conditions()

odes, free_vars, param_dict, constraints = asp_model.generate_ode_system()
ode_dict = {var: ode for var, ode in zip(default_initial_conditions, odes)}
print('\n\n')
pprint(ode_dict)


default initial conditions:
{'ARF_1': 1.0,
 'ARF_2': 1.0,
 'D__ARF_1__ARF_1': 0.1,
 'D__ARF_1__ARF_2': 0.1,
 'D__ARF_1__iaa_1': 0.1,
 'D__ARF_2__ARF_2': 0.1,
 'D__ARF_2__iaa_1': 0.1,
 'D__iaa_1__iaa_1': 0.1,
 'R__iaa_1': 0.0,
 'TAS3': 0.1,
 'iaa_1': 0.1,
 'miR390': 0.1,
 'tasi_R_ARF': 1.0}
Model state variables
[{'name': 'ARF_1', 'description': '', 'components': ()}, {'name': 'ARF_2', 'description': '', 'components': ()}, {'name': 'D__ARF_1__ARF_1', 'description': 'Dimer of ARF_1 protein and ARF_1 protein', 'components': ('ARF_1', 'ARF_1')}, {'name': 'D__ARF_1__ARF_2', 'description': 'Dimer of ARF_1 protein and ARF_2 protein', 'components': ('ARF_1', 'ARF_2')}, {'name': 'D__ARF_1__iaa_1', 'description': 'Dimer of ARF_1 protein and iaa_1 protein', 'components': ('ARF_1', 'iaa_1')}, {'name': 'D__ARF_2__ARF_2', 'description': 'Dimer of ARF_2 protein and ARF_2 protein', 'components': ('ARF_2', 'ARF_2')}, {'name': 'D__ARF_2__iaa_1', 'description': 'Dimer of ARF_2 protein and iaa_1 protein',

In [ ]:
solver = asp_model.make_forward_solver()

rhs_func = asp_model.generate_rhs_func()

rhs_func(default_initial_conditions.values(), 0, default_parameters.values(), [1.0])
jac_func = asp_model.generate_jacobian_func()
jac_func(default_initial_conditions.values(), 0, default_parameters.values(), [1.0])


ts = np.linspace(0, 10, 10)

aux_level = 1.0
default_output, success = solver(ts, default_parameters,
                            default_initial_conditions,
                            1.0)

import matplotlib.pyplot as plt
plt.plot(ts, default_output / default_output.max(axis=1)[:, None],
         label=list(default_initial_conditions))
plt.legend()
plt.show()

In [ ]:
custom_parameters = \
{'d__arf__all': 1.0,
 'k__arf__all': 1.0,
 'd__D__ARF_1__ARF_1': 0.1,
 'd__D__ARF_1__iaa_1': 0.1,
 'd__D__iaa_1__iaa_1': 100,
 'd__G__ARF_1__ARF_1': 0.01,
 'd__G__ARF_1__iaa_1': 0.01,
 'd__R': 0.004166666666666667,
 'd__iaa_1__x': 5000,
 'd__iaa__all': 1.0,
 'k__D__ARF_1__ARF_1': 0.25,
 'k__D__ARF_1__iaa_1': 0.25,
 'k__D__iaa_1__iaa_1': 0.001,
 'k__G__ARF_1__ARF_1': 0.1,
 'k__G__ARF_1__ARF_1__R__iaa_1': 1.0,
 'k__G__ARF_1__R__iaa_1': 1.0,
 'k__G__ARF_1__iaa_1': 0.1,
 'k__G__ARF_1__iaa_1__R__iaa_1': 1.0,
 'k__G__R__iaa_1': 1.0,
}

ts = np.linspace(0, 1000, 250)

all_params = asp_model.get_default_parameters()

for k, v in custom_parameters.items():
    all_params[k] = v

output, success = solver(ts, all_params,
                            default_initial_conditions,
                            1.0)
import matplotlib.pyplot as plt
from itertools import cycle
lines = ["-","--","-.",":"]
linecycler = cycle(lines)
# output = output / output[:, i].max()

for i in range(output.shape[1]):
    plt.plot(ts, output[:, i]/output[:, i].max() , label=list(default_initial_conditions)[i],
            linestyle=next(linecycler))
plt.legend()
plt.show()


In [ ]:
asp_model.output_julia()